# Linear Regression

**Objective:** fit a straight-line model `Y = wX + b` to data, first
on the exact deterministic example from the slides (inches -> cm,
slides 12-13), then on a noisier, more realistic example (years of
experience -> salary), evaluating it with MSE / RMSE (slides 60-61).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

## Part 1: Centimeters from Inches

Slides 12-13 use `A * Inch + B = Cm` as the running example of what a
model "learns" - here, `A` should come out to `2.54` and `B` to `0`,
since 1 inch is exactly 2.54 cm.

In [ ]:
inches = np.array([1, 2, 3, 4, 5, 6, 7, 8]).reshape(-1, 1)
cm = inches * 2.54

model_cm = LinearRegression()
model_cm.fit(inches, cm)

print('Learned A (coefficient):', model_cm.coef_[0][0])
print('Learned B (intercept):  ', model_cm.intercept_[0])

The model recovered the exact formula from the data, with no
prior knowledge of the conversion rate - this is what slide 13 means
by a "trained model capable of generalizing the relationship... for
any new observations." 

## Part 2: A Noisier Example - Salary vs. Years of Experience

Real data is rarely a perfect line. We simulate salaries with some
random noise around a true underlying trend, then see how well
Linear Regression can recover it.

In [ ]:
rng = np.random.default_rng(42)
years_experience = np.linspace(0, 15, 60)
noise = rng.normal(loc=0, scale=4000, size=60)
salary = 35000 + 4500 * years_experience + noise

salary_df = pd.DataFrame({'YearsExperience': years_experience, 'Salary': salary})
salary_df.head()

### Step 1: Split into train / test sets

We hold out 20% of the data to check how well the model generalizes
to points it never saw during training.

In [ ]:
X = salary_df[['YearsExperience']]
y = salary_df['Salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('train size:', len(X_train), '| test size:', len(X_test))

### Step 2: Fit the model

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

print('Learned slope (w):    ', lr_model.coef_[0])
print('Learned intercept (b):', lr_model.intercept_)

### Step 3: Evaluate with MSE / RMSE

Slide 60 defines `MSE = (1/n) * sum((Y - Y_hat)^2)` - the closer the
predictions are to the actual values, the smaller the error. RMSE is
just the square root of MSE, back in the original units (dollars,
here).

In [ ]:
y_pred = lr_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print('MSE: ', round(mse, 2))
print('RMSE:', round(rmse, 2))

### Step 4: Visualize the best-fit line

Slide 61 asks "which line is the best predictor?" and compares lines
by their RMSE. Let's plot our fitted line against the test points.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, label='training data', alpha=0.6)
plt.scatter(X_test, y_test, label='test data', color='orange')
plt.plot(X, lr_model.predict(X), color='red', label='fitted line')
plt.xlabel('Years of Experience')
plt.ylabel('Salary')
plt.legend()
plt.title(f'Linear Regression (RMSE = {rmse:,.0f})')
plt.show()

The slope (`w`) tells us how much salary increases per additional
year of experience, and the intercept (`b`) is the model's predicted
starting salary at zero years of experience - these are the
"coefficients and parameters" slide 11 compares to tuning knobs on a
synthesizer.